In [3]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

In [4]:
df = pd.read_csv('/Users/nic/Documents/MM2/data/ftc_labeled.csv')
print(df.shape)
df.head()

(339, 20)


,case_name,url,date,case_type,matter_number,case_status,summary,tags,long_title,statutes,penalty_usd,press_release_urls,pdf_urls,press_release_text,penalty_usd_enriched,text_len,severity_tier,statutes_recovered,statutes_final,violation_labels
0,"Amazon.com, Inc., U.S. v.",https://www.ftc.gov/legal-library/browse/cases...,2026-06-30,Federal,2523024,Pending,Amazon will pay $2.25 million in civil penalti...,Consumer Protection; Bureau of Consumer Protec...,"UNITED STATES OF AMERICA, Plaintiff v. AMAZON....",FCRA,2250000.0,https://www.ftc.gov/news-events/news/press-rel...,https://www.ftc.gov/system/files/ftc_gov/pdf/A...,Amazon will pay $2.25 million in civil penalti...,2250000.0,4522.0,Medium,FCRA,FCRA,['FCRA']
1,"FTC v Kochava, Inc.",https://www.ftc.gov/legal-library/browse/cases...,2026-06-26,Federal,NaN,Pending,The FTC will prohibit data broker Kochava and ...,Consumer Protection; Bureau of Consumer Protec...,"Federal Trade Commission, Plaintiff, V. Kochav...",NaN,NaN,https://www.ftc.gov/news-events/news/press-rel...,https://www.ftc.gov/system/files/ftc_gov/pdf/1...,The Federal Trade Commission filed a lawsuit a...,NaN,8807.0,NaN,HBNR,HBNR,['Other']
2,"Illuminate Education, Inc., In the Matter of",https://www.ftc.gov/legal-library/browse/cases...,2026-06-05,Administrative,2223105,Under Order,The Federal Trade Commission will require educ...,Consumer Protection; Bureau of Consumer Protec...,"In the Matter of ILLUMINATE EDUCATION, INC., a...",NaN,NaN,https://www.ftc.gov/news-events/news/press-rel...,https://www.ftc.gov/system/files/ftc_gov/pdf/2...,The Federal Trade Commission will require educ...,NaN,7304.0,NaN,NaN,FTC Act Section 5,['Section 5']
3,"Twitter, Inc., a corporation",https://www.ftc.gov/legal-library/browse/cases...,2026-06-03,Administrative,0923093,NaN,NaN,Consumer Protection; Bureau of Consumer Protec...,"In the Matter of Twitter, Inc.,a corporation",NaN,NaN,https://www.ftc.gov/news-events/news/press-rel...,https://www.ftc.gov/sites/default/files/docume...,Social networking service Twitter has agreed t...,NaN,10506.0,NaN,NaN,FTC Act Section 5,['Section 5']
4,"CMG Media Corporation, In the Matter of",https://www.ftc.gov/legal-library/browse/cases...,2026-05-21,Administrative,2423029,Pending,"The FTC will require Cox Media Group, MindSift...",Consumer Protection; Bureau of Consumer Protec...,In the Matter of CMG Media Corporation,NaN,930000.0,https://www.ftc.gov/news-events/news/press-rel...,https://www.ftc.gov/system/files/ftc_gov/pdf/C...,The Federal Trade Commission will require Cox ...,930000.0,4967.0,Low,FTC Act Section 5,FTC Act Section 5,['Section 5']


In [5]:
import ast
df['violation_labels'] = df['violation_labels'].apply(lambda x: ast.literal_eval(x) if pd.notna(x) else [])
print(df['violation_labels'].head())

0         [FCRA]
1        [Other]
2    [Section 5]
3    [Section 5]
4    [Section 5]
Name: violation_labels, dtype: object


In [6]:
# drop rows with no text or no labels
model_df = df[df['press_release_text'].notna() & df['violation_labels'].apply(len).gt(0)].copy()
print(model_df.shape)

# binarize labels
mlb = MultiLabelBinarizer()
Y = mlb.fit_transform(model_df['violation_labels'])
print('Classes:', mlb.classes_)
print('Y shape:', Y.shape)

(333, 20)
Classes: ['COPPA' 'FCRA' 'GLBA' 'Other' 'Privacy Shield' 'Section 5']
Y shape: (333, 6)


In [8]:
tfidf = TfidfVectorizer(max_features=5000, stop_words='english')
X = tfidf.fit_transform(model_df['press_release_text'])
print('X shape:', X.shape)

X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=777)
print('Train:', X_train.shape, 'Test:', X_test.shape)


X shape: (333, 5000)
Train: (266, 5000) Test: (67, 5000)


In [10]:
clf = OneVsRestClassifier(LogisticRegression(max_iter=1000, random_state=777, class_weight='balanced'))
clf.fit(X_train, Y_train)

Y_pred = clf.predict(X_test)
print(classification_report(Y_test, Y_pred, target_names=mlb.classes_))

                precision    recall  f1-score   support

         COPPA       1.00      1.00      1.00         9
          FCRA       0.83      0.83      0.83        12
          GLBA       0.57      0.57      0.57         7
         Other       1.00      0.33      0.50         6
Privacy Shield       1.00      0.88      0.94        17
     Section 5       0.67      0.91      0.77        22

     micro avg       0.80      0.82      0.81        73
     macro avg       0.85      0.75      0.77        73
  weighted avg       0.83      0.82      0.81        73
   samples avg       0.81      0.86      0.82        73



/Users/nic/privacy-env/lib/python3.14/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in samples with no predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


In [11]:
severity_df = df[df['press_release_text'].notna() & df['severity_tier'].notna()].copy()
print(severity_df.shape)
print(severity_df['severity_tier'].value_counts())

(147, 20)
severity_tier
Low       71
Medium    49
High      27
Name: count, dtype: int64


In [12]:
from sklearn.preprocessing import LabelEncoder

X_sev = tfidf.transform(severity_df['press_release_text'])

le = LabelEncoder()
Y_sev = le.fit_transform(severity_df['severity_tier'])
print('Classes:', le.classes_)

X_train_s, X_test_s, Y_train_s, Y_test_s = train_test_split(X_sev, Y_sev, test_size=0.2, random_state=42, stratify=Y_sev)
print('Train:', X_train_s.shape, 'Test:', X_test_s.shape)

Classes: ['High' 'Low' 'Medium']
Train: (117, 5000) Test: (30, 5000)


In [13]:
from sklearn.linear_model import LogisticRegression

clf_sev = LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced')
clf_sev.fit(X_train_s, Y_train_s)

Y_pred_s = clf_sev.predict(X_test_s)
print(classification_report(Y_test_s, Y_pred_s, target_names=le.classes_, zero_division=0))

              precision    recall  f1-score   support

        High       0.38      0.50      0.43         6
         Low       0.83      0.71      0.77        14
      Medium       0.60      0.60      0.60        10

    accuracy                           0.63        30
   macro avg       0.60      0.60      0.60        30
weighted avg       0.66      0.63      0.64        30



In [14]:
!pip install transformers torch -q

In [15]:
from transformers import AutoTokenizer, AutoModel
import torch

tokenizer = AutoTokenizer.from_pretrained('nlpaueb/legal-bert-base-uncased')
model = AutoModel.from_pretrained('nlpaueb/legal-bert-base-uncased')
print('Legal-BERT loaded')

config.json:   0%|          | 0.00/1.02k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/222k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: nlpaueb/legal-bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Legal-BERT loaded


In [16]:
def get_embedding(text, max_length=512):
    inputs = tokenizer(text, return_tensors='pt', truncation=True, max_length=max_length, padding=True)
    with torch.no_grad():
        outputs = model(**inputs)
    return outputs.last_hidden_state[:, 0, :].squeeze().numpy()

print('Testing embedding shape:', get_embedding(severity_df['press_release_text'].iloc[0]).shape)

Testing embedding shape: (768,)


In [17]:
print('Generating embeddings...')
embeddings = []
for i, text in enumerate(severity_df['press_release_text']):
    embeddings.append(get_embedding(text))
    if i % 20 == 0:
        print(f'{i}/{len(severity_df)}')

X_bert = np.array(embeddings)
print('Done. Shape:', X_bert.shape)

Generating embeddings...
0/147
20/147
40/147
60/147
80/147
100/147
120/147
140/147
Done. Shape: (147, 768)


In [18]:
X_train_b, X_test_b, Y_train_b, Y_test_b = train_test_split(X_bert, Y_sev, test_size=0.2, random_state=42, stratify=Y_sev)

clf_bert_sev = LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced')
clf_bert_sev.fit(X_train_b, Y_train_b)

Y_pred_b = clf_bert_sev.predict(X_test_b)
print(classification_report(Y_test_b, Y_pred_b, target_names=le.classes_, zero_division=0))

              precision    recall  f1-score   support

        High       0.57      0.67      0.62         6
         Low       0.75      0.64      0.69        14
      Medium       0.55      0.60      0.57        10

    accuracy                           0.63        30
   macro avg       0.62      0.64      0.63        30
weighted avg       0.65      0.63      0.64        30



In [20]:
def get_embedding_firstlast(text):
    tokens = tokenizer(text, return_tensors='pt', truncation=False, add_special_tokens=False)
    input_ids = tokens['input_ids'][0]
    if len(input_ids) > 510:
        input_ids = torch.cat([input_ids[:255], input_ids[-255:]])
    input_ids = torch.cat([
        torch.tensor([tokenizer.cls_token_id]),
        input_ids,
        torch.tensor([tokenizer.sep_token_id])
    ]).unsqueeze(0)
    attention_mask = torch.ones_like(input_ids)
    with torch.no_grad():
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
    return outputs.last_hidden_state[:, 0, :].squeeze().numpy()

In [21]:
# first+last 256 token embeddings
embeddings_fl = []
for i, text in enumerate(severity_df['press_release_text']):
    embeddings_fl.append(get_embedding_firstlast(text))
    if i % 20 == 0:
        print(f'{i}/{len(severity_df)}')

X_bert_fl = np.array(embeddings_fl)

X_train_fl, X_test_fl, Y_train_fl, Y_test_fl = train_test_split(X_bert_fl, Y_sev, test_size=0.2, random_state=42, stratify=Y_sev)

clf_fl = LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced')
clf_fl.fit(X_train_fl, Y_train_fl)

Y_pred_fl = clf_fl.predict(X_test_fl)
print('First+Last 256 tokens:')
print(classification_report(Y_test_fl, Y_pred_fl, target_names=le.classes_, zero_division=0))

0/147
20/147
40/147
60/147
80/147
100/147
120/147
140/147
First+Last 256 tokens:
              precision    recall  f1-score   support

        High       0.50      0.50      0.50         6
         Low       0.79      0.79      0.79        14
      Medium       0.60      0.60      0.60        10

    accuracy                           0.67        30
   macro avg       0.63      0.63      0.63        30
weighted avg       0.67      0.67      0.67        30



In [22]:
all_tags = df['tags'].str.split(';').explode().str.strip()
print(sorted(all_tags.unique()))

['Advertising and Marketing', 'Advertising and Marketing Basics', 'Alcohol', 'Artificial Intelligence', 'Automobiles', 'Big Data', 'Bureau of Consumer Protection', 'Cars', 'Children', "Children's Online Privacy Protection Act (COPPA)", "Children's Privacy", 'Clothing and Textiles', 'Consumer Privacy', 'Consumer Protection', 'Credit & Loan Offers', 'Credit Reporting', 'Credit and Finance', 'Credit and Loans', 'Data Privacy Framework', 'Data Security', 'Debt', 'Debt Collection', 'Debt Relief', 'Do Not Call', 'Education', 'Endorsements, Influencers, and Reviews', 'Entertainment', 'Fair Credit Reporting Act (FCRA)', 'Fake Check', 'FinTech', 'Finance', 'Franchises, Business Opportunities, and Investments', 'Gaming', 'Gramm-Leach-Bliley Act', 'Health', 'Health Care', 'Health Claims', 'Health Privacy', 'Health Professional Services', 'Human Resources', 'Identity Theft', 'Internet of Things', 'Merchandise & Clothing', 'Mobile', 'Mortgages', 'Northwest Region', 'Office of Technology Research an